# Notebook 02 — Prepare Your Manufacturing Data

**What you’ll learn:** How to set up Unity Catalog tables that Genie will query. This is the foundation — Genie can only answer questions about data it can see.

**What happens:**
1. Creates a Unity Catalog **schema** for the workshop tables.
2. Loads data from the Parquet files created by notebook 01.
3. Casts string dates to proper DATE types.
4. Creates **SQL functions** that Genie can call for complex calculations.
5. Adds **table comments** so Genie understands what each column means.

**Before you start:** Run notebook **01** first.

**Compute:** Serverless.

## Compute

Use **Serverless** (recommended) or a Unity Catalog–enabled cluster.

## Load config

Your catalog and schema are set in `00_workshop_config`. This cell loads them.

In [ ]:
%run ./00_workshop_config

## Install dependencies

On **serverless**, `%pip` installs into the notebook environment; `restartPython()` clears the interpreter so the next cells pick up new packages.


In [ ]:
%pip install databricks-sdk==0.74.0 --quiet

## Restart Python after pip install

In [ ]:
dbutils.library.restartPython()

## Re-load config after Python restart

Python restart clears all variables, so we re-run the config notebook.

In [ ]:
%run ./00_workshop_config

## Step 1: Create Unity Catalog schema

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Using {CATALOG}.{SCHEMA}")

## Step 2: Load data from notebook 01

Reads the Parquet files created by notebook 01 and writes them as Delta tables in your schema.

In [ ]:
fqn = f"{CATALOG}.{SCHEMA}"

TABLE_NAMES = [
    "plants", "production_lines", "operators", "production_events",
    "quality_metrics_daily", "safety_incidents", "equipment_feedback",
]

print(f"Loading data from: {PREBUILD_PATH}")
for table_name in TABLE_NAMES:
    parquet_path = f"{PREBUILD_PATH}/{table_name}"
    df = spark.read.parquet(parquet_path)
    spark.sql(f"DROP TABLE IF EXISTS {fqn}.{table_name}")
    df.write.mode("overwrite").saveAsTable(f"{fqn}.{table_name}")
    count = spark.table(f"{fqn}.{table_name}").count()
    print(f"  {table_name}: {count} rows")

print("\nAll 7 tables loaded.")

## Step 3: Cast date columns

Cast string date columns to proper DATE types so filters and dashboards work correctly.

In [ ]:
fqn = f"{CATALOG}.{SCHEMA}"

spark.sql(f"""
CREATE OR REPLACE TABLE {fqn}.production_events AS
SELECT event_id, event_type, CAST(event_date AS DATE) AS event_date,
       event_timestamp, production_line_id, operator_id, part_number, unit_serial_vin, defect_code
FROM {fqn}.production_events
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {fqn}.production_lines AS
SELECT line_id, line_name, description, product_type, plant_id,
       daily_capacity, cost_per_unit,
       CAST(start_date AS DATE) AS start_date,
       CAST(end_date AS DATE) AS end_date, status
FROM {fqn}.production_lines
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {fqn}.quality_metrics_daily AS
SELECT CAST(date AS DATE) AS date, plant_id, production_line_id,
       units_produced, units_passed_inspection, defects_found,
       downtime_minutes, scrap_count, rework_count,
       first_pass_yield, oee_score
FROM {fqn}.quality_metrics_daily
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {fqn}.safety_incidents AS
SELECT incident_id, description, severity, production_line_id,
       CAST(incident_date AS DATE) AS incident_date,
       resolution_status, category, root_cause, corrective_action
FROM {fqn}.safety_incidents
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {fqn}.operators AS
SELECT operator_id, first_name, last_name, shift, certification,
       CAST(hire_date AS DATE) AS hire_date, plant_id
FROM {fqn}.operators
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {fqn}.equipment_feedback AS
SELECT feedback_id, equipment_name, comment,
       CAST(feedback_date AS DATE) AS feedback_date,
       production_line_id, operator_id
FROM {fqn}.equipment_feedback
""")

print("✓ All date columns cast to DATE type")

## Step 4: Create analytics functions

Build three reusable SQL table-valued functions for defect rate analysis, OEE computation, and best production line lookup. These functions are available to Genie for natural language queries.

In [ ]:
fqn = f"{CATALOG}.{SCHEMA}"

spark.sql(f"""
CREATE OR REPLACE FUNCTION {fqn}.compute_defect_rate(
  line_id STRING COMMENT 'Production line ID, e.g. L001',
  start_dt DATE  COMMENT 'Start date, e.g. 2024-01-01',
  end_dt DATE    COMMENT 'End date, e.g. 2024-06-30'
)
RETURNS TABLE (
  production_line STRING,
  total_units INT,
  total_defects INT,
  defect_rate DOUBLE
)
COMMENT 'Returns defect rate for a production line over a date range'
RETURN
  SELECT
    pl.line_name AS production_line,
    COUNT(CASE WHEN pe.event_type = 'unit_produced' THEN 1 END) AS total_units,
    COUNT(CASE WHEN pe.event_type = 'defect_detected' THEN 1 END) AS total_defects,
    ROUND(COUNT(CASE WHEN pe.event_type = 'defect_detected' THEN 1 END) * 100.0 /
      NULLIF(COUNT(CASE WHEN pe.event_type = 'unit_produced' THEN 1 END), 0), 2) AS defect_rate
  FROM {fqn}.production_events pe
  JOIN {fqn}.production_lines pl ON pe.production_line_id = pl.line_id
  WHERE pe.production_line_id = line_id
    AND pe.event_date BETWEEN start_dt AND end_dt
  GROUP BY pl.line_name
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {fqn}.compute_oee_summary(
  p_id STRING  COMMENT 'Plant ID, e.g. P001',
  start_dt DATE COMMENT 'Start date, e.g. 2024-01-01',
  end_dt DATE   COMMENT 'End date, e.g. 2024-06-30'
)
RETURNS TABLE (
  plant_name STRING,
  avg_oee DOUBLE,
  avg_first_pass_yield DOUBLE,
  total_downtime_hours DOUBLE
)
COMMENT 'Returns OEE summary for a plant over a date range'
RETURN
  SELECT
    p.plant_name,
    ROUND(AVG(qm.oee_score) * 100, 2) AS avg_oee,
    ROUND(AVG(qm.first_pass_yield) * 100, 2) AS avg_first_pass_yield,
    ROUND(SUM(qm.downtime_minutes) / 60.0, 1) AS total_downtime_hours
  FROM {fqn}.quality_metrics_daily qm
  JOIN {fqn}.plants p ON qm.plant_id = p.plant_id
  WHERE qm.plant_id = p_id
    AND qm.date BETWEEN start_dt AND end_dt
  GROUP BY p.plant_name
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {fqn}.get_best_production_line()
RETURNS TABLE (
  production_line STRING,
  product_type STRING,
  avg_first_pass_yield DOUBLE,
  total_units_produced INT
)
COMMENT 'Returns the production line with the highest first pass yield'
RETURN
  SELECT
    pl.line_name AS production_line,
    pl.product_type,
    ROUND(AVG(qm.first_pass_yield) * 100, 2) AS avg_first_pass_yield,
    SUM(qm.units_produced) AS total_units_produced
  FROM {fqn}.quality_metrics_daily qm
  JOIN {fqn}.production_lines pl ON qm.production_line_id = pl.line_id
  GROUP BY pl.line_name, pl.product_type
  ORDER BY avg_first_pass_yield DESC
  LIMIT 1
""")

print("✓ Created functions: compute_defect_rate, compute_oee_summary, get_best_production_line")

## Step 5: Table comments for Genie

Short **table-level** comments help Genie and analysts understand grain and domain. Adjust text if you extend the model.

In [ ]:
fqn = f"{CATALOG}.{SCHEMA}"
comments = {
    "plants": "Manufacturing sites with capacity and revenue context.",
    "production_lines": "Assembly/paint/stamping/EV lines; join to plants and events.",
    "operators": "Shop-floor operators, shift, certifications, home plant.",
    "production_events": "Event grain: unit_produced, defect_detected, scrap, downtime, rework, inspection. unit_serial_vin is a 17-char unit traceability id (sensitive).",
    "quality_metrics_daily": "Daily OEE (0-1), first_pass_yield (0-1), downtime_minutes per line.",
    "safety_incidents": "Safety events tied to production_line_id with severity.",
    "equipment_feedback": "Operator comments on equipment by line.",
}
for t, cmt in comments.items():
    esc = cmt.replace("'", "''")
    spark.sql(f"ALTER TABLE {fqn}.{t} SET TBLPROPERTIES ('comment' = '{esc}')")
print("Applied table comments.")


## Setup complete

You now have **7 Delta tables** and **3 SQL functions** (`compute_defect_rate`, `compute_oee_summary`, `get_best_production_line`).

**Next:** run `03_create_genie_spaces` to create the **blank** and **configured** Genie spaces and save their IDs to `workshop_config`.